In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [ ]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any, NamedTuple
from typing_extensions import Protocol, runtime_checkable
from pprint import pprint

from src.dataset import get_iter
from src.datasets.sum import Addition
from src.decoding import make_autoregressive
from src.rollout import rollout
from src.utils import parse_dict
from src.verifier import make_compute_returns

In [ ]:
dataset = Addition(
    10,
    8,
    False,
    0,
    sequence_type="question_only",
    right_to_left=True,
    carry_registers=True,
    match_carry=True,
)

In [ ]:
data_iter = dataset.get_sequences()

In [ ]:
batch = next(data_iter)
batch

In [ ]:
checkpoint = dill.load(open(
    "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results/addition-max_int_64-no_eos/metastable_ppo-scratch_1M-0_cot_tokens-8x8-11-18-25_09_03_56-deacc19b-7394-432c-aca2-8d5d3592c247/models/1000000.dill",
    "rb",
))

In [ ]:
from penzai import pz

import IPython

pz.ts.register_as_default()

# Optional automatic array visualization extras:
pz.ts.register_autovisualize_magic()
pz.enable_interactive_context()
pz.ts.active_autovisualizer.set_interactive(pz.ts.ArrayAutovisualizer())

checkpoint.params["embedders"]["token_emb"]["embedding"].value @ checkpoint.params["embedders"]["token_emb"]["embedding"].value.T

## Debug

In [ ]:
base_path = "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results"
algo_name = "addition-max_int_64-no_eos"
run_name = "metastable_ppo-scratch_1M-0_cot_tokens-8x8-11-18-25_09_03_56-deacc19b-7394-432c-aca2-8d5d3592c247"

learner_path = os.path.join(base_path, algo_name, run_name)

config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))
config_dict["gamma"] = 0.99
config = parse_dict(config_dict)
dataset_kwargs = config.dataset_kwargs
half_precision = config.half_precision
dtype = jnp.bfloat16 if half_precision else jnp.float32

checkpoint_i = 0

num_evals = 1
max_decode_len = 100
train_val_ratio = 0.8
max_batch_size = 2
eval_seed = 42
num_bits = 8

curr_int = 2 ** num_bits
num_pairs = curr_int * curr_int
num_val = num_pairs - int(np.floor(num_pairs * train_val_ratio))
batch_size = min(num_val, max_batch_size)

Dtype = Any
Shape = tuple[int, ...]

if config.dataset_name == "curriculum":
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.datasets[-1]["dataset_kwargs"]["max_int"])))
    train_val_ratio = dataset_kwargs.datasets[-1]["dataset_kwargs"]["train_val_ratio"]
    predict_eos = dataset_kwargs.datasets[-1]["dataset_kwargs"]["predict_eos"]
    num_cot_tokens = dataset_kwargs.datasets[-1]["dataset_kwargs"]["num_cot_tokens"]
    right_to_left = dataset_kwargs.datasets[-1]["dataset_kwargs"]["right_to_left"]
    correctness_aware = dataset_kwargs.datasets[-1]["dataset_kwargs"]["correctness_aware"]
    carry_registers = dataset_kwargs.datasets[-1]["dataset_kwargs"]["carry_registers"]
else:
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.max_int)))
    train_val_ratio = dataset_kwargs.train_val_ratio
    predict_eos = dataset_kwargs.predict_eos
    num_cot_tokens = dataset_kwargs.num_cot_tokens
    right_to_left = dataset_kwargs.right_to_left
    correctness_aware = dataset_kwargs.correctness_aware
    carry_registers = dataset_kwargs.carry_registers

dataset = Addition(
    context_len=max_decode_len,
    max_int=curr_int,
    train=True,
    seed=eval_seed,
    sequence_type="question_only",
    train_val_ratio=train_val_ratio,
    right_to_left=right_to_left,
    num_repeats=None,
    shuffle=False,
    exact=True,
    predict_eos=predict_eos,
    num_cot_tokens=num_cot_tokens,
    p_inject_noop=0.0,
    max_noops=0,
    noop_as_pad=False,
    p_curriculum=0.99,
    reverse_curriculum=False,
    correctness_aware=correctness_aware,
    carry_registers=carry_registers,
)

compute_returns = make_compute_returns(config, dataset.eos_token_id, dataset.reset_token_id, dataset.token_map)

print(f"Max int: {curr_int}, batch size: {batch_size}")

out_dim = int(dataset.output_space.n)
data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
)
data_iter = get_iter(data_loader, None, dtype)
batch = next(data_iter)
batch = {
    "sequence": np.repeat(batch["sequence"], num_evals, axis=0),
    "mask": np.repeat(batch["mask"], num_evals, axis=0),
    "target": np.repeat(batch["target"], num_evals, axis=0),
    "pointer_correct": np.repeat(batch["pointer_correct"], num_evals, axis=0),
}

In [ ]:
# Load model
last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[checkpoint_i]
train_state = dill.load(
    open(os.path.join(learner_path, "models", last_step), "rb")
)

# print(train_state.params)
model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)

rng = jax.random.PRNGKey(44)
rng, rollout_rng = jax.random.split(rng)

# Sample next batch and evaluate
_, init_cache = make_autoregressive(
    model,
    max_decode_len=max_decode_len,
    batch_size=batch_size * num_evals,
    embed_dim=config_dict["model_config"]["model_kwargs"]["embed_dim"],
    dtype=dtype,
    eval_mode=True,
)
cache = init_cache()
graphdef, _, rest = nnx.split(model, nnx.Cache, ...)

rollout_res = rollout(
    graphdef,
    cache,
    rest,
    rollout_rng,
    batch,
    eos_token=dataset.eos_token_id,
    deterministic=0,
    correct_aware_shift=dataset.correctness_aware_tokens_offset,
    max_token_id_to_shift=dataset.max_token_id_to_shift,
)

rollout_res = {
    "observations": rollout_res[0],
    "actions": rollout_res[1],
    "eos": rollout_res[2],
    "question_mask": rollout_res[3],
    "last_prompt_idxes": rollout_res[4],
}

In [ ]:
rollout_res["observations"]

In [ ]:
rollout_res["actions"]

In [ ]:
print(batch["target"][0])

In [ ]:
new_batch = {
    "sequence": batch["sequence"],
    "target": batch["target"],
    "observations": rollout_res["observations"],
    "actions": rollout_res["actions"],
}
returns_info = compute_returns(
    new_batch,
    rollout_res["last_prompt_idxes"],
    is_eval=False,
)

In [ ]:
idx = 0

print([int(token) for token in batch["mask"][idx]])
print(rollout_res["last_prompt_idxes"][idx])
print([int(token) for token in batch["sequence"][idx]])
question_len = int(np.where(batch["sequence"][idx] == 3)[0][0]) - 1
print([9] * question_len + [int(token) for token in batch["target"][idx]][:-question_len])
print([int(token) for token in rollout_res["observations"][idx]])
print([9] * question_len + [int(token) for token in rollout_res["actions"][idx]][question_len + 1:] + [9])

In [ ]:
idx = 0
try:
    pprint({k: v[idx][:-1] if k != "actions" else v[idx][1:] for k, v in new_batch.items()})
except:
    pprint(new_batch)

In [ ]:
returns_info[1][idx], returns_info[2][idx], returns_info[0][idx]

In [ ]:
returns_info[1], returns_info[2], returns_info[0]

TODO:
- We can have input size where efficiently using the context length can solve